## 🎯 Learning Objectives
* Design and implement a multi-agent pipeline using LangChain's latest features.
* Define distinct roles and assign appropriate tools to individual agents within a collaborative workflow.
* Integrate LangSmith for comprehensive observability, tracing, and debugging of multi-agent interactions.
* Understand how to orchestrate sequential agent execution using LangChain Expression Language (LCEL).


## Exercise: Building a Multi-Agent Pipeline with LangSmith Observability

### Task Description

Your goal is to build a multi-agent system designed to perform **Market Research and generate a concise Report** for a hypothetical new product launch. This system should consist of at least two distinct agents working collaboratively to achieve the final output. Crucially, your solution must be fully observable using LangSmith.

#### Scenario: New Product Market Research

Imagine your company is launching a new AI-powered personal assistant. You need to understand the current market, identify key competitors, and summarize potential challenges and opportunities. Your multi-agent system will automate this process.

#### Agent Roles & Responsibilities:

1.  **Market Researcher Agent**: This agent is responsible for gathering raw information. It should use a search tool to find data on market trends, competitor products, and user needs related to AI personal assistants.
2.  **Report Writer Agent**: This agent takes the raw research findings from the Market Researcher and synthesizes them into a structured, concise market research report. The report should include an executive summary, key findings, competitor overview, and recommendations.

### Requirements:

*   **Two Distinct Agents**: Implement at least two `AgentExecutor` instances (or similar LangChain agent constructs) with clearly defined roles as described above.
*   **Tool Usage**: The Market Researcher Agent *must* utilize a search tool (e.g., `TavilySearchResults` or a mock equivalent if you don't have an API key). The Report Writer Agent will primarily use its LLM capabilities for synthesis, but you may optionally give it a simple 'file writer' tool if you wish to simulate saving the report.
*   **Multi-Agent Orchestration**: Design a workflow where the output of the Market Researcher Agent serves as the primary input for the Report Writer Agent. Use LangChain Expression Language (LCEL) for chaining the agents.
*   **LangSmith Observability**: Ensure that your entire multi-agent pipeline is traceable in LangSmith. This includes setting up the necessary environment variables and observing the distinct steps and agent calls within the LangSmith UI.
*   **Clear Prompts**: Each agent should have a well-defined system prompt that guides its behavior and output format.
*   **Output**: The final output should be the generated market research report.

### Evaluation Criteria:

*   **Correctness**: Does the pipeline successfully execute and produce a relevant report based on the initial query?
*   **Agent Design**: Are the agent roles clear, and do they effectively utilize their assigned tools and prompts?
*   **LangSmith Integration**: Is the entire workflow, including individual agent steps and tool calls, clearly visible and traceable in LangSmith? (Demonstrate by providing a LangSmith trace URL if possible).
*   **Code Quality**: Is the code clean, well-commented, and easy to understand? Does it follow modern LangChain best practices (e.g., LCEL)?
*   **Robustness**: Does the solution handle typical inputs gracefully?


In [ ]:
# Setup Code

import os
from typing import List, Dict, Any

# Ensure you have these installed: pip install langchain langchain-openai langchain-community tavily-python
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_community.tools.tavily_research import TavilySearchResults
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# --- Environment Variables (Replace with your actual keys or set in your environment) ---
# For OpenAI LLM
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")
# For Tavily Search Tool
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY", "YOUR_TAVILY_API_KEY")

# --- LangSmith Configuration (Crucial for observability) ---
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "YOUR_LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "AG04-L10-MultiAgent-Exercise") # Name your project

# --- Initialize LLM (GPT-4o is a good choice for 2026) ---
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

# --- Define Tools ---
# 1. Tavily Search Tool for Market Researcher Agent
tavily_search_tool = TavilySearchResults(max_results=5)

# 2. (Optional) Mock File Writer Tool for Report Writer Agent
@tool
def write_report_to_file(report_content: str, filename: str = "market_research_report.md") -> str:
    """Writes the given report content to a markdown file."""
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(report_content)
        return f"Report successfully written to {filename}"
    except Exception as e:
        return f"Error writing report to file: {e}"

# List of tools for the Market Researcher Agent
market_researcher_tools = [tavily_search_tool]

# List of tools for the Report Writer Agent (can be empty or include the file writer)
report_writer_tools = [write_report_to_file]

print("Setup complete. LLM and tools initialized. LangSmith tracing enabled.")
print(f"LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")


### Your Implementation

Now it's your turn! Implement the multi-agent pipeline as described in the task. Use the provided setup code for the LLM and tools. Focus on creating clear agent prompts, defining the agents, and orchestrating their interaction using LCEL. Remember to leverage LangSmith for observability.

Your solution should define:

1.  **`market_researcher_agent_executor`**: An `AgentExecutor` for the Market Researcher.
2.  **`report_writer_agent_executor`**: An `AgentExecutor` for the Report Writer.
3.  **`multi_agent_pipeline`**: An LCEL chain that connects these two agents.

Finally, run your `multi_agent_pipeline` with a sample query and print the final report.


In [ ]:
# --- Reference Solution ---

# 1. Define Prompts for each Agent

# Market Researcher Agent Prompt
market_researcher_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert Market Researcher. Your goal is to gather comprehensive information on a given topic using the provided search tools. Focus on market trends, competitor analysis, and user needs. Summarize your findings clearly and concisely, ready for a report writer to process."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

# Report Writer Agent Prompt
report_writer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a professional Report Writer. Your task is to take raw research findings and synthesize them into a well-structured, concise market research report. The report should include an Executive Summary, Key Findings, Competitor Overview, and Recommendations. Ensure the language is professional and easy to understand. Format the report using Markdown."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "Here are the research findings: {research_findings}\n\nNow, please generate the market research report."),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

# 2. Create Agents

# Market Researcher Agent
market_researcher_agent = create_openai_tools_agent(
    llm=llm,
    tools=market_researcher_tools,
    prompt=market_researcher_prompt,
)
market_researcher_agent_executor = AgentExecutor(
    agent=market_researcher_agent,
    tools=market_researcher_tools,
    verbose=True, # Set to True to see agent's thought process in console
    handle_parsing_errors=True,
    max_iterations=5,
    # Assign a name for LangSmith tracing clarity
    name="MarketResearcherAgent"
)

# Report Writer Agent
report_writer_agent = create_openai_tools_agent(
    llm=llm,
    tools=report_writer_tools, # Can be empty if no tools are strictly needed
    prompt=report_writer_prompt,
)
report_writer_agent_executor = AgentExecutor(
    agent=report_writer_agent,
    tools=report_writer_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3,
    # Assign a name for LangSmith tracing clarity
    name="ReportWriterAgent"
)

# 3. Orchestrate the Multi-Agent Pipeline using LCEL

# Define the input for the pipeline
initial_query = {"input": "Research the market for a new AI-powered personal assistant, focusing on current trends, key competitors, and user needs."}

# The pipeline: Researcher -> Report Writer
multi_agent_pipeline = (
    RunnablePassthrough.assign(  # Pass the initial input to the researcher
        research_findings=RunnableLambda(lambda x: market_researcher_agent_executor.invoke({"input": x["input"]})["output"])
    )
    | RunnableLambda(lambda x: report_writer_agent_executor.invoke({"research_findings": x["research_findings"]})["output"])
)

# 4. Run the Pipeline
print("\n--- Running Multi-Agent Pipeline ---\n")
final_report = multi_agent_pipeline.invoke(initial_query)

print("\n--- Final Market Research Report ---\n")
print(final_report)

# To view the trace in LangSmith:
# 1. Ensure your LANGCHAIN_API_KEY and LANGCHAIN_PROJECT are set correctly.
# 2. Visit https://smith.langchain.com/ and navigate to your project (e.g., 'AG04-L10-MultiAgent-Exercise').
# 3. You should see a new run corresponding to this execution, showing the sequential calls to 'MarketResearcherAgent' and 'ReportWriterAgent'.
